In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/patientpunk.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


# Judgement ⑥ — coreference: does the model use upstream context?

The weakest judgement to validate — the frozen run had zero coreference items (a parent-id defect), so we probe it directly by **context ablation**: classify each (comment, drug) with its parent context and without. A model that resolves coreference should change its answer when the context is removed on **context-necessary** items (the drug is *only* in the parent), and *not* on **control** items (the drug is already in the comment, so the context is redundant — the noise floor). The gap between the two is the coreference signal.

In [ ]:

import json
import numpy as np
from collections import defaultdict
d=json.load(open(r"C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/validation/j6_coreference_runs.json")); R=[r for r in d["records"] if not r["parse_failed"]]
short=lambda m:m.split("/")[-1]
# pair up with/without per (model, sample, drug)
byk=defaultdict(dict)
for r in R: byk[(r["model"], r["sample_id"], r["drug"], r["drug_in_comment"])][r["variant"]]=(r["sentiment"], r["signal"])
def changed(v): return v.get("with") is not None and v.get("without") is not None and v["with"]!=v["without"]
MODELS=sorted(set(r["model"] for r in R))
display(Markdown(f"*(loaded — {len(R)} classifications, {len(MODELS)} models; "
 f"{d['manifest']['n_context_necessary']} context-necessary pairs)*"))


## 1. Does removing context change the answer — necessary vs control?

Per model: how often the sentiment/signal flips when the parent context is ablated, on context-necessary items vs control items.

In [ ]:

rows=[]
for m in MODELS:
    nec=[changed(v) for (mm,s,dr,dic),v in byk.items() if mm==m and dic==False and "with" in v and "without" in v]
    ctl=[changed(v) for (mm,s,dr,dic),v in byk.items() if mm==m and dic==True  and "with" in v and "without" in v]
    if nec: rows.append((short(m), np.mean(nec), (np.mean(ctl) if ctl else 0), len(nec), len(ctl)))
rows.sort(key=lambda x:-(x[1]-x[2]))
import numpy as np
y=np.arange(len(rows)); h=0.38
fig,ax=plt.subplots(figsize=(9,7))
ax.barh(y+h/2,[r[1]*100 for r in rows],h,label="context-necessary (drug only in parent)",color="#2a78d6")
ax.barh(y-h/2,[r[2]*100 for r in rows],h,label="control (drug in comment — noise floor)",color="#c0392b")
ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows],fontsize=8); ax.invert_yaxis()
ax.set_xlabel("change-rate when parent context is removed (%)"); ax.legend(fontsize=8)
ax.set_title("Coreference probe — answer changes on context-necessary vs control items"); fig.tight_layout(); plt.show()
nec_all=np.mean([r[1] for r in rows]); ctl_all=np.mean([r[2] for r in rows])
display(Markdown(f"**Context-necessary items change {nec_all:.0%} of the time when the parent is removed, vs "
 f"{ctl_all:.0%} for control items** — a **{nec_all-ctl_all:+.0%}** gap in the expected direction, but a *weak* "
 f"one. Two reasons to read it cautiously: the control change-rate is **{ctl_all:.0%}** — removing *redundant* "
 f"context shouldn't move the answer at all, so the models are noticeably unstable to prompt edits, and the "
 f"coreference signal sits only ~5 points above that noise floor. And the context-necessary group is tiny "
 f"(**16 pairs**), so the gap is suggestive, not conclusive. **Best read: a hint that models use the reply "
 f"chain, not proof — ⑥ stays the least-validated judgement.**"))


## 2. Which models depend on context most

In [ ]:

display(Markdown(
 "Models with a **large necessary-minus-control gap** resolve coreference strongly (they lean on the parent "
 "when the drug isn't local). A **small gap** means either the model ignores upstream context (a coreference "
 "weakness) or it guesses locally regardless. A model whose *control* change-rate is high is unstable — its "
 "answer wobbles even when context is redundant, which is noise, not coreference. Read the two bars together: "
 "the healthy pattern is a tall blue bar and a short red one."))
tb=[[r[0], f"{r[1]:.0%}", f"{r[2]:.0%}", f"{(r[1]-r[2]):+.0%}"] for r in rows]
display(HTML(pd.DataFrame(tb, columns=["model","necessary Δ-rate","control Δ-rate","gap (coreference signal)"]).to_html(index=False)))


## 3. Verdict

In [ ]:

nec_all=np.mean([r[1] for r in rows]); ctl_all=np.mean([r[2] for r in rows])
lines=[
 f"- **Weak-positive, not a clean pass.** The gap is in the right direction (**{nec_all:.0%}** necessary vs "
 f"**{ctl_all:.0%}** control, **{nec_all-ctl_all:+.0%}**), so models *lean* on the reply chain — but the effect "
 f"is small and rides on only 16 context-necessary pairs. ⑥ remains the least-validated of the eleven.",
 f"- **The {ctl_all:.0%} control change-rate is a red flag of its own** — removing redundant context shouldn't "
 "change the answer, so the classifier is fairly unstable to prompt edits (temperature is pinned to 0, so this "
 "is prompt-sensitivity, not sampling). That instability, not coreference, may be the bigger reliability story.",
 "- **We could only ever show *use*, not *correctness*** — ablation shows the model depends on context, never "
 "that it resolves it right. With no coreference gold and a defect-zeroed frozen run, this probe on real items "
 "is the honest ceiling for ⑥.",
 "- **Recommendation:** widen the context-necessary set (construct thread-structure items) and drive down the "
 "control change-rate (prompt-stability check) before treating coreference as validated.",
]
display(Markdown("\n".join(lines)))


## Limitations

- **Change ≠ correctness** — ablation shows the model *depends on* context, not that its resolution is right.
  With no coreference gold, "uses context" is the strongest claim available (the plan's stated ceiling for ⑥).
- **Context-necessary is inferred** from whether the drug string appears in the comment text — a surface
  heuristic; a drug referred to only by pronoun/synonym in the comment could be mislabelled.
- **Depends on the parent_context field** in the IRR frame (280/300 pairs) at depth 2; deeper chains untested.
- Measures whether a resolution step fires, not any treatment effect. Not medical advice.

In [ ]:

M=d["manifest"]
prov=pd.DataFrame({"item":["method","models","pairs","context-necessary","metric","skill"],
 "value":[M["method"], str(len(MODELS)), str(M["n_pairs"]), str(M["n_context_necessary"]),
          "change-rate under context ablation (necessary vs control)", "research-assistant v2"]})
display(HTML("<b>Provenance</b>"+prov.to_html(index=False)))
display(HTML('<div style="font-size:1.15em;font-weight:bold;font-style:italic;margin-top:1em">'
             'Measures whether a resolution step fires, not treatment effects. Not medical advice.</div>'))
